# 07 - Pilot Active-Window Analysis

Controlled analysis only. This notebook derives candidate Hijack/Leak active windows from locally available detector-specific observations. It does not download MRT files, create Normal labels, extract features, train models, alter partitions, or change the frozen roster.

## Interpretation Guardrails

- ACTIVE means detector-supported incident activity.
- SILENT/UNKNOWN does not mean Normal.
- Hijack/Leak first-to-last detector spans are not assumed continuous.
- Outage windows remain documented Cloudflare intervals and are not changed here.

In [ ]:
from pathlib import Path
import csv
import sys
from collections import Counter

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent.resolve()
sys.path.insert(0, str(ROOT / 'src'))

from bgp_incident_detection.active_windows import run

def read_csv(path):
    with path.open(newline='', encoding='utf-8') as handle:
        return list(csv.DictReader(handle))

## Generate Active-Window Artifacts

In [ ]:
results = run()
print('Observations:', len(results['observations']))
print('Bins:', Counter(row['activity_status'] for row in results['bins']))
print('Gap rows:', len(results['gap_analysis']))
print('Threshold rows:', len(results['sensitivity']))
print('Segment rows:', len(results['segments']))

## Event-Specific Observation Timeline

In [ ]:
read_csv(ROOT / 'data/pilot/pilot_event_observation_timeline_v1.csv')

## Gap Analysis

In [ ]:
read_csv(ROOT / 'data/pilot/pilot_active_gap_analysis_v1.csv')

## Threshold Sensitivity

In [ ]:
read_csv(ROOT / 'data/pilot/pilot_active_window_threshold_sensitivity_v1.csv')

## Candidate Active Segments

In [ ]:
read_csv(ROOT / 'data/pilot/pilot_candidate_active_segments_v1.csv')

## Volume Comparison

In [ ]:
volume = read_csv(ROOT / 'data/pilot/pilot_active_window_volume_comparison_v1.csv')
print('Original Hijack/Leak files:', sum(int(row['original_planned_file_count']) for row in volume))
print('Candidate active-window files:', sum(int(row['active_window_planned_file_count']) for row in volume))
volume

## Recommendation

The local Cloudflare detector records for these pilot events contain boundary observations and aggregate counts, not dense event-observation timelines. Therefore, the threshold sensitivity is useful for acquisition planning, but it is not sufficient to freeze a final training label-window rule.